In [4]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Book Rating Analysis") \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/31 16:55:08 WARN Utils: Your hostname, Ataollahs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.4 instead (on interface en0)
26/08/31 16:55:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/31 16:55:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [12]:
rate_df = spark.read.csv("data/Book-Ratings.csv", header=True, inferSchema=True,sep=";")
type(rate_df)

pyspark.sql.classic.dataframe.DataFrame

In [13]:
rate_df.printSchema()

root
 |-- userid: integer (nullable = true)
 |-- isbn: string (nullable = true)
 |-- rate: integer (nullable = true)



In [14]:
rate_df.show()

+------+----------+----+
|userid|      isbn|rate|
+------+----------+----+
|276725|034545104X|   0|
|276726|0155061224|   5|
|276727|0446520802|   0|
|276729|052165615X|   3|
|276729|0521795028|   6|
|276733|2080674722|   0|
|276736|3257224281|   8|
|276737|0600570967|   6|
|276744|038550120X|   7|
|276745| 342310538|  10|
|276746|0425115801|   0|
|276746|0449006522|   0|
|276746|0553561618|   0|
|276746|055356451X|   0|
|276746|0786013990|   0|
|276746|0786014512|   0|
|276747|0060517794|   9|
|276747|0451192001|   0|
|276747|0609801279|   0|
|276747|0671537458|   9|
+------+----------+----+
only showing top 20 rows


In [16]:
rate_df.show(5)

+------+----------+----+
|userid|      isbn|rate|
+------+----------+----+
|276725|034545104X|   0|
|276726|0155061224|   5|
|276727|0446520802|   0|
|276729|052165615X|   3|
|276729|0521795028|   6|
+------+----------+----+
only showing top 5 rows


In [19]:
df2 = rate_df.where("rate > 5")

In [21]:
df2.show(5)

+------+----------+----+
|userid|      isbn|rate|
+------+----------+----+
|276729|0521795028|   6|
|276736|3257224281|   8|
|276737|0600570967|   6|
|276744|038550120X|   7|
|276745| 342310538|  10|
+------+----------+----+
only showing top 5 rows


In [22]:
df2.count()

363268

In [24]:
rate_df.count()

1149780

In [25]:
rate_df.filter("rate > 5").select("userid","rate").show(5)

+------+----+
|userid|rate|
+------+----+
|276729|   6|
|276736|   8|
|276737|   6|
|276744|   7|
|276745|  10|
+------+----+
only showing top 5 rows


In [27]:
rate_df.groupBy("isbn").count().show(5)

+----------+-----+
|      isbn|count|
+----------+-----+
|2080674722|    3|
|3499134004|    1|
|3548603203|   24|
|880781112X|    3|
|0738205737|    3|
+----------+-----+
only showing top 5 rows


In [28]:
rate_df.groupBy("userid").avg("rate").show(5)

+------+-----------------+
|userid|        avg(rate)|
+------+-----------------+
|278515|              0.0|
|   463|              8.0|
|   496|5.333333333333333|
|  1238|              0.0|
|  1591|             10.0|
+------+-----------------+
only showing top 5 rows


In [41]:
from pyspark.sql.functions import *
rate_df.groupBy("isbn")\
    .agg(count("rate"),sum("rate"),max("rate"),avg("rate"),stddev("rate"))\
        .orderBy(col("avg(rate)").desc())\
            .show(10)

+----------+-----------+---------+---------+---------+------------+
|      isbn|count(rate)|sum(rate)|max(rate)|avg(rate)|stddev(rate)|
+----------+-----------+---------+---------+---------+------------+
|0837850711|          1|       10|       10|     10.0|        NULL|
|1587361175|          1|       10|       10|     10.0|        NULL|
|0963335545|          1|       10|       10|     10.0|        NULL|
|0374250995|          1|       10|       10|     10.0|        NULL|
|1410798860|          1|       10|       10|     10.0|        NULL|
|076530418X|          1|       10|       10|     10.0|        NULL|
|0877732248|          1|       10|       10|     10.0|        NULL|
|1591822076|          1|       10|       10|     10.0|        NULL|
|039397281X|          2|       20|       10|     10.0|         0.0|
|1573245003|          1|       10|       10|     10.0|        NULL|
+----------+-----------+---------+---------+---------+------------+
only showing top 10 rows


In [42]:
rate_df.groupBy("isbn")\
    .agg(count("rate"),sum("rate"),max("rate"),avg("rate"),stddev("rate"))\
        .where("avg(rate) > 5")\
        .orderBy(col("avg(rate)").desc())\
            .show(10)

+----------+-----------+---------+---------+---------+------------+
|      isbn|count(rate)|sum(rate)|max(rate)|avg(rate)|stddev(rate)|
+----------+-----------+---------+---------+---------+------------+
|0837850711|          1|       10|       10|     10.0|        NULL|
|1587361175|          1|       10|       10|     10.0|        NULL|
|0963335545|          1|       10|       10|     10.0|        NULL|
|0374250995|          1|       10|       10|     10.0|        NULL|
|1410798860|          1|       10|       10|     10.0|        NULL|
|076530418X|          1|       10|       10|     10.0|        NULL|
|0877732248|          1|       10|       10|     10.0|        NULL|
|1591822076|          1|       10|       10|     10.0|        NULL|
|039397281X|          2|       20|       10|     10.0|         0.0|
|1573245003|          1|       10|       10|     10.0|        NULL|
+----------+-----------+---------+---------+---------+------------+
only showing top 10 rows


In [45]:
rate_df.createOrReplaceTempView("rate_df")
spark.sql("select isbn, rate from rate_df").show(5)

+----------+----+
|      isbn|rate|
+----------+----+
|034545104X|   0|
|0155061224|   5|
|0446520802|   0|
|052165615X|   3|
|0521795028|   6|
+----------+----+
only showing top 5 rows


In [46]:
rate_df.write.json("rate_df.json")

In [47]:
rate_df.describe().show()

+-------+------------------+-----------+------------------+
|summary|            userid|       isbn|              rate|
+-------+------------------+-----------+------------------+
|  count|           1149780|    1149780|           1149780|
|   mean|140386.39512602412|   Infinity|2.8669501991685364|
| stddev|  80562.2777185127|        NaN|3.8541838592016617|
|    min|                 2| 0330299891|                 0|
|    max|            278854| �423350229|                10|
+-------+------------------+-----------+------------------+



In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
schema = StructType([
    StructField("user", StringType(), True),
    StructField("isbn", StringType(), True),
    StructField("rate", FloatType(), True)
])

